In [3]:
import os
import shutil
from pathlib import Path
import pandas as pd

# =========================
# Paths
# =========================
csv_path = Path("573_Gloss_List.csv")

source_1 = Path(r"D:\150 words sample\cleaned_landmarks_147words")
source_2 = Path(r"D:\ASL Citizen\data\raw_landmarks")

output_dir = Path("raw_landmarks")
output_dir.mkdir(exist_ok=True)

# =========================
# Load gloss list
# =========================
df = pd.read_csv(csv_path)
glosses = df["Gloss"].astype(str).str.upper().tolist()

# =========================
# Helpers
# =========================
def find_files(gloss, folder):
    matches = []

    if not folder.exists():
        return matches

    for file in folder.rglob("*.npy"):
        stem = file.stem  # filename without .npy

        # strict rule: must start with "GLOSS " then number
        if stem.startswith(gloss + " "):
            rest = stem[len(gloss) + 1:]

            # ensure remaining part is ONLY a number (strict safety)
            if rest.isdigit():
                matches.append(file)

    return matches

# =========================
# Main processing
# =========================
results = []
missing = []

for gloss in glosses:
    gloss = gloss.strip().upper()

    files = find_files(gloss, source_1)

    # only if NOT found in first source, check second
    if not files:
        files = find_files(gloss, source_2)

    if not files:
        missing.append(gloss)
        continue

    copied_names = []

    for f in files:
        dest = output_dir / f.name

        # avoid overwrite conflicts
        if dest.exists():
            dest = output_dir / f"{f.stem}_{gloss}.npy"

        shutil.copy2(f, dest)
        copied_names.append(dest.name)

    results.append({
        "gloss": gloss,
        "count": len(files),
        "files_names": ";".join(copied_names)
    })

# =========================
# Save outputs
# =========================
final_df = pd.DataFrame(results)
final_df.to_csv("final_gloss_files.csv", index=False)

missing_df = pd.DataFrame({"missing_gloss": missing})
missing_df.to_csv("missing_glosses.csv", index=False)

print("Done!")
print(f"Copied glosses: {len(final_df)}")
print(f"Missing glosses: {len(missing)}")

Done!
Copied glosses: 501
Missing glosses: 72
